In [3]:
import os, shutil, datetime, itertools
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
from pathlib import Path
from google.colab import drive
import keras_tuner as kt


In [2]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 7.6 MB/s eta 0:00:00


In [4]:

# ---------------- Mount & Paths ----------------
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DRIVE_ROOT}/DL/Emotion_EfficientNet_NaiveBayes_Tuned_70_30_{ts}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(DATA_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(DATA_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)


Mounted at /content/drive


In [5]:
# ---------------- Data pipeline ----------------
IMG_SIZE_RAW = (48, 48)    # dataset images (grayscale)
IMG_SIZE_INP = (224, 224)  # model input
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

def make_loader(dir_path, shuffle=True, batch=BATCH):
    ds = image_dataset_from_directory(
        dir_path,
        labels="inferred",
        label_mode="categorical",   # one-hot for Keras metrics/ROC
        color_mode="grayscale",
        image_size=IMG_SIZE_RAW,
        batch_size=batch,
        shuffle=shuffle,
        seed=SEED
    )
    class_names = ds.class_names

    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        x = tf.image.resize(x, IMG_SIZE_INP)
        x = tf.cast(x, tf.float32) / 255.0  # keep [0,1]; branch-wise preprocess later
        return x, y

    return ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE), class_names

train_ds, class_names = make_loader(LOCAL_TRAIN, shuffle=True)
val_ds, _             = make_loader(LOCAL_VAL,   shuffle=False)
test_ds, _            = make_loader(LOCAL_TEST,  shuffle=False)
num_classes = len(class_names)
print("Classes:", class_names, "| Num classes:", num_classes)

Found 28000 files belonging to 7 classes.
Found 8616 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised'] | Num classes: 7


In [6]:

# ---------------- EfficientNet Feature Extraction ----------------
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_pp

# EfficientNetB0 model (pre-trained)
efficientnet_base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
efficientnet_base.trainable = False

inp = layers.Input(shape=(224,224,3), name="inp_rgb01")  # [0,1] RGB

# EfficientNet preprocess & features
x = layers.Lambda(lambda z: efficientnet_pp(z*255.0), name="efficientnet_preproc")(inp)
f = efficientnet_base(x, training=False)
f = layers.GlobalAveragePooling2D(name="gap_efficientnet")(f)

# Classifier head (Naive Bayes)
out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

model = models.Model(inp, out, name="EfficientNet_NaiveBayes_Hybrid")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "EfficientNet_NaiveBayes_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inp_rgb01 (InputLayer)          │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnet_preproc (Lambda)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap_efficientnet                │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pred (Dense)                    │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,058,538 (15.48 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [7]:
# ---------------- Hyperparameter Tuning Function ----------------

def build_model(hp):
    efficientnet_base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
    efficientnet_base.trainable = False

    inp = layers.Input(shape=(224,224,3), name="inp_rgb01")

    # EfficientNet preprocess & features
    x = layers.Lambda(lambda z: efficientnet_pp(z*255.0), name="efficientnet_preproc")(inp)
    f = efficientnet_base(x, training=False)
    f = layers.GlobalAveragePooling2D(name="gap_efficientnet")(f)

    # Hyperparameter tuning for the Dense layer and Dropout
    f = layers.Dense(
        hp.Int('dense_units', min_value=256, max_value=1024, step=128),
        activation="relu"
    )(f)

    f = layers.BatchNormalization()(f)
    f = layers.Dropout(hp.Float('dropout_rate', min_value=0.3, max_value=0.7, step=0.1))(f)

    out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

    model = models.Model(inp, out, name="EfficientNet_NaiveBayes_Hybrid")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-3, sampling='LOG')),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ---------------- Keras Tuner ----------------
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=2,  # More epochs for better tuning
    hyperband_iterations=2,  # Number of iterations for tuning
    directory=OUT_DIR,
    project_name="efficientnet_naivebayes_tuning"
)


In [8]:

# ---------------- Tuning the Model ----------------
tuner.search(train_ds, validation_data=val_ds, epochs=10, batch_size=BATCH)

# Get the best model
best_model = tuner.get_best_models()[0]
best_model.summary()


Trial 4 Complete [00h 02m 14s]
val_accuracy: 0.39461466670036316

Best val_accuracy So Far: 0.5068477392196655
Total elapsed time: 00h 09m 13s


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "EfficientNet_NaiveBayes_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inp_rgb01 (InputLayer)          │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnet_preproc (Lambda)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap_efficientnet                │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     1,311,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pred (Dense)                    │ (None, 7)              │         7,175 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,372,586 (20.49 MB)

 Trainable params: 1,320,967 (5.04 MB)

 Non-trainable params: 4,051,619 (15.46 MB)

In [9]:
# ---------------- Save the Best Model ----------------
best_model.save(os.path.join(OUT_DIR, "efficientnet_naivebayes_best_tuned.keras"))

# ---------------- Training the Best Model ----------------
history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,  # After tuning, train for more epochs
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(os.path.join(OUT_DIR, "best_model_tuned.keras"), save_best_only=True),
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
)

# Save the final model and training history
best_model.save(os.path.join(OUT_DIR, "efficientnet_naivebayes_final_tuned.keras"))
pd.DataFrame(history.history).to_csv(os.path.join(OUT_DIR, "training_history_tuned.csv"), index=False)


Epoch 1/5
875/875 ━━━━━━━━━━━━━━━━━━━━ 84s 70ms/step - accuracy: 0.4838 - loss: 1.4563 - val_accuracy: 0.5230 - val_loss: 1.3179
Epoch 2/5
875/875 ━━━━━━━━━━━━━━━━━━━━ 49s 55ms/step - accuracy: 0.5052 - loss: 1.3654 - val_accuracy: 0.5311 - val_loss: 1.2994
Epoch 3/5
875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.5311 - loss: 1.3005 - val_accuracy: 0.5346 - val_loss: 1.2826
Epoch 4/5
875/875 ━━━━━━━━━━━━━━━━━━━━ 49s 56ms/step - accuracy: 0.5419 - loss: 1.2640 - val_accuracy: 0.5417 - val_loss: 1.2702
Epoch 5/5
875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.5567 - loss: 1.2145 - val_accuracy: 0.5477 - val_loss: 1.2532


In [11]:
# ---------------- Curves ----------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy Curve"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves_tuned.png"), dpi=150)
plt.close()
print("Training curves saved.")

# ---------------- Evaluate on Test ----------------
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"TEST -> loss: {test_loss:.4f} | acc: {test_acc:.4f}")

# ---------------- Predictions & Reports ----------------
y_true = []
for _, y in test_ds.unbatch():
    y_true.append(y.numpy())
y_true = np.array(y_true)
y_true_cls = np.argmax(y_true, axis=1)

y_prob = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Classification Report
rep = classification_report(y_true_cls, y_pred, target_names=class_names, digits=4)
print(rep)
with open(os.path.join(OUT_DIR, "classification_report_tuned.txt"), "w") as f:
    f.write(rep)

# Confusion Matrix
cm = confusion_matrix(y_true_cls, y_pred)
plt.figure(figsize=(7.2,6.2))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix"); plt.colorbar()
ticks = np.arange(num_classes)
plt.xticks(ticks, class_names, rotation=45)
plt.yticks(ticks, class_names)
plt.ylim(num_classes-0.5, -0.5) # Corrected y-axis limits
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix_tuned.png"), dpi=150)
plt.close()
print("Confusion matrix saved.")

# ROC Curve (example for one class)
# For a multi-class problem, you might want to plot ROC for each class or micro/macro average
# This requires converting y_true_cls to one-hot or using OneVsRestClassifier approach with scikit-learn
# Here's an example for one class (e.g., 'happy')
try:
    happy_class_index = class_names.index('happy')
    fpr, tpr, _ = roc_curve(y_true[:, happy_class_index], y_prob[:, happy_class_index])
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic for class: happy')
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(OUT_DIR, "roc_curve_happy_tuned.png"), dpi=150)
    plt.close()
    print("ROC curve for 'happy' class saved.")
except ValueError:
    print("Class 'happy' not found in class_names. Skipping ROC curve generation.")

Training curves saved.
TEST -> loss: 1.2511 | acc: 0.5442
              precision    recall  f1-score   support

       angry     0.4731    0.3768    0.4195       958
   disgusted     0.3333    0.4144    0.3695       111
     fearful     0.4395    0.2305    0.3024      1024
       happy     0.6648    0.7982    0.7254      1774
     neutral     0.4866    0.5442    0.5138      1233
         sad     0.4400    0.4643    0.4518      1247
   surprised     0.6525    0.7184    0.6838       831

    accuracy                         0.5442      7178
   macro avg     0.4985    0.5067    0.4952      7178
weighted avg     0.5308    0.5442    0.5300      7178

Confusion matrix saved.
ROC curve for 'happy' class saved.
